In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.models import Model

from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("train.csv")

df['comment_text'] = df['comment_text'].astype(str)

labels = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']

texts = df['comment_text'].tolist()
y = df[labels].values

print("Dataset size:", len(texts))

Dataset size: 159571


In [4]:
N = 3  # number of previous messages

contexts = []

for i in range(len(texts)):
    prev = texts[max(0, i-N):i]
    context = " [SEP] ".join(prev)
    contexts.append(context)

data = pd.DataFrame({
    "comment": texts,
    "context": contexts
})

for col in labels:
    data[col] = df[col]

print(data.head())

                                             comment  \
0  Explanation\nWhy the edits made under my usern...   
1  D'aww! He matches this background colour I'm s...   
2  Hey man, I'm really not trying to edit war. It...   
3  "\nMore\nI can't make any real suggestions on ...   
4  You, sir, are my hero. Any chance you remember...   

                                             context  toxic  severe_toxic  \
0                                                         0             0   
1  Explanation\nWhy the edits made under my usern...      0             0   
2  Explanation\nWhy the edits made under my usern...      0             0   
3  Explanation\nWhy the edits made under my usern...      0             0   
4  D'aww! He matches this background colour I'm s...      0             0   

   obscene  threat  insult  identity_hate  
0        0       0       0              0  
1        0       0       0              0  
2        0       0       0              0  
3        0       0      

In [5]:
MAX_WORDS = 20000
MAX_COMMENT_LEN = 150
MAX_CONTEXT_LEN = 200

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(data['comment'])

In [6]:
X_comment = tokenizer.texts_to_sequences(data['comment'])
X_context = tokenizer.texts_to_sequences(data['context'])

X_comment = pad_sequences(X_comment, maxlen=MAX_COMMENT_LEN, padding='post')
X_context = pad_sequences(X_context, maxlen=MAX_CONTEXT_LEN, padding='post')

y = data[labels].values

In [7]:
Xc_train, Xc_val, Xctx_train, Xctx_val, y_train, y_val = train_test_split(
    X_comment, X_context, y, test_size=0.2, random_state=42
)

In [8]:
y_binary = (y_train.sum(axis=1) > 0).astype(int)

neg = np.sum(y_binary == 0)
pos = np.sum(y_binary == 1)

print("Non-toxic:", neg)
print("Toxic:", pos)

weight_for_1 = neg / pos

sample_weights = np.where(y_binary == 1, weight_for_1, 1.0)

Non-toxic: 114675
Toxic: 12981


In [9]:
# Inputs
comment_input = Input(shape=(MAX_COMMENT_LEN,), name='comment')
context_input = Input(shape=(MAX_CONTEXT_LEN,), name='context')

# Shared embedding
embedding = Embedding(MAX_WORDS + 1, 128)

# Comment branch
x1 = embedding(comment_input)
x1 = Bidirectional(LSTM(128))(x1)
x1 = Dropout(0.4)(x1)

# Context branch
x2 = embedding(context_input)
x2 = Bidirectional(LSTM(128))(x2)
x2 = Dropout(0.4)(x2)

# Merge
merged = Concatenate()([x1, x2])

x = Dense(64, activation='relu')(merged)
x = Dropout(0.3)(x)

output = Dense(6, activation='sigmoid')(x)

model = Model(inputs=[comment_input, context_input], outputs=output)

In [10]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=[tf.keras.metrics.AUC(name='auc')]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ comment             │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context             │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 128)  │  2,560,128 │ comment[0][0],    │
│ (Embedding)         │                   │            │ context[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 256)       │    263,168 │ embedding[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 256)       │    263,168 │ embedding[1][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 512)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │     32,832 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 6)         │        390 │ dropout_2[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,119,686 (11.90 MB)

 Trainable params: 3,119,686 (11.90 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(
    {'comment': Xc_train, 'context': Xctx_train},
    y_train,
    validation_data=({'comment': Xc_val, 'context': Xctx_val}, y_val),
    epochs=5,
    batch_size=256,
    sample_weight=sample_weights
)

Epoch 1/5
499/499 ━━━━━━━━━━━━━━━━━━━━ 393s 782ms/step - auc: 0.9338 - loss: 0.4682 - val_auc: 0.9767 - val_loss: 0.0688
Epoch 2/5
499/499 ━━━━━━━━━━━━━━━━━━━━ 448s 898ms/step - auc: 0.9747 - loss: 0.3508 - val_auc: 0.9793 - val_loss: 0.0607
Epoch 3/5
499/499 ━━━━━━━━━━━━━━━━━━━━ 436s 874ms/step - auc: 0.9833 - loss: 0.3103 - val_auc: 0.9795 - val_loss: 0.0685
Epoch 4/5
499/499 ━━━━━━━━━━━━━━━━━━━━ 442s 886ms/step - auc: 0.9879 - loss: 0.2749 - val_auc: 0.9740 - val_loss: 0.0606
Epoch 5/5
499/499 ━━━━━━━━━━━━━━━━━━━━ 2986s 6s/step - auc: 0.9914 - loss: 0.2376 - val_auc: 0.9705 - val_loss: 0.0675


In [12]:
def predict(text, context=""):
    seq_c = tokenizer.texts_to_sequences([text])
    seq_ctx = tokenizer.texts_to_sequences([context])
    
    pad_c = pad_sequences(seq_c, maxlen=MAX_COMMENT_LEN, padding='post')
    pad_ctx = pad_sequences(seq_ctx, maxlen=MAX_CONTEXT_LEN, padding='post')
    
    pred = model.predict({'comment': pad_c, 'context': pad_ctx})[0]
    
    return {label: float(round(p, 3)) for label, p in zip(labels, pred)}

In [13]:
print("\nNo context:")
print(predict("you idiot"))

print("\nFriendly context:")
print(predict("you're dead to me", "haha bro you're amazing 😂"))

print("\nHostile context:")
print(predict("you're dead to me", "I will find you and hurt you"))


No context:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 487ms/step
{'toxic': 0.9959999918937683, 'severe_toxic': 0.10599999874830246, 'obscene': 0.9210000038146973, 'threat': 0.029999999329447746, 'insult': 0.8809999823570251, 'identity_hate': 0.1289999932050705}

Friendly context:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
{'toxic': 0.9419999718666077, 'severe_toxic': 0.0, 'obscene': 0.017000000923871994, 'threat': 0.0020000000949949026, 'insult': 0.054999999701976776, 'identity_hate': 0.003000000026077032}

Hostile context:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
{'toxic': 0.9449999928474426, 'severe_toxic': 0.0, 'obscene': 0.04699999839067459, 'threat': 0.003000000026077032, 'insult': 0.11800000071525574, 'identity_hate': 0.006000000052154064}


In [14]:
t1 = predict("you're dead to me")['toxic']
t2 = predict("you're dead to me", "haha bro you're amazing 😂")['toxic']
t3 = predict("you're dead to me", "I will find you and hurt you")['toxic']

print(t1, t2, t3)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
0.9300000071525574 0.9419999718666077 0.9449999928474426
